# API Project - Bus Bunching
### Multi-agency Comparsion (WMATA, MBTA, MTA)
Question: What route and temporal factors are associated with bus bunching on high-frequency corridors, and do these relationships vary across different transit agencies (NYC, Boston, and DC)?

In [12]:
import pandas as pd
import numpy as np
import requests
import ratelim
import tenacity

### 1) WMATA API
#### Bus Routes: D40, D60, D80, C53, C61, D20

In [13]:
WMATA_KEY = "5ec08fcd21744590b1a0f655710c4c91"

In [14]:
wmata_url = "https://api.wmata.com/Bus.svc/json/jBusPositions"
wmata_params = {"RouteID": "D40"}  
wmata_headers = {"api_key": WMATA_KEY}

requests.get(wmata_url, params=wmata_params, headers=wmata_headers)

<Response [200]>

In [15]:
wmata_response = requests.get(wmata_url, params=wmata_params, headers=wmata_headers)
wmata_response.json()

{'BusPositions': [{'VehicleID': '5514',
   'Lat': 38.99368836449795,
   'Lon': -77.03037011818807,
   'Deviation': 5.0,
   'DateTime': '2026-03-26T14:17:01',
   'TripID': '30634020',
   'RouteID': 'D40',
   'DirectionNum': 0,
   'DirectionText': 'NORTH',
   'TripHeadsign': 'SILVER SPRING',
   'TripStartTime': '2026-03-26T13:12:00',
   'TripEndTime': '2026-03-26T14:09:00',
   'BlockNumber': 'M618'},
  {'VehicleID': '5533',
   'Lat': 38.89541855980368,
   'Lon': -77.02191162109375,
   'Deviation': 2.0,
   'DateTime': '2026-03-26T14:16:42',
   'TripID': '25675020',
   'RouteID': 'D40',
   'DirectionNum': 1,
   'DirectionText': 'SOUTH',
   'TripHeadsign': 'ARCHIVES',
   'TripStartTime': '2026-03-26T13:12:00',
   'TripEndTime': '2026-03-26T14:18:00',
   'BlockNumber': 'M608'},
  {'VehicleID': '5505',
   'Lat': 38.91591188019397,
   'Lon': -77.02191925048828,
   'Deviation': 6.0,
   'DateTime': '2026-03-26T14:16:39',
   'TripID': '33133020',
   'RouteID': 'D40',
   'DirectionNum': 1,
   'Dir

#### Vars Used: 
- `Deviation` — the primary outcome, minutes late/early so no GTFS needed
- `DateTime` — timestamp will be used for peak hour flag
- `RouteID`, `DirectionNum`, `VehicleID`, `TripID` — route/trip identifiers (`DirectionNum`, 0=North 1=South)
- `TripStartTime` — scheduled headway, will be used to calc the gap between consecutive trips
- `Lat`, `Lon` — Vehicle position

In [16]:
wmata_fields = {
    'VehicleID': 'vehicle_id',
    'Lat': 'lat',
    'Lon': 'lon',
    'Deviation': 'deviation',
    'DateTime': 'timestamp',
    'TripID': 'trip_id',
    'RouteID': 'route_id',
    'DirectionNum': 'direction',
    'TripStartTime': 'trip_start_time'
}

wmata_positions = wmata_response.json()['BusPositions']

df = pd.DataFrame(wmata_positions)[list(wmata_fields.keys())].rename(columns=wmata_fields)
df['agency'] = 'WMATA'

print(df)
print(df.dtypes) #Checks for var types (timestamp and trip_start_time)

   vehicle_id        lat        lon  deviation            timestamp   trip_id  \
0        5514  38.993688 -77.030370        5.0  2026-03-26T14:17:01  30634020   
1        5533  38.895419 -77.021912        2.0  2026-03-26T14:16:42  25675020   
2        5505  38.915912 -77.021919        6.0  2026-03-26T14:16:39  33133020   
3        5525  38.961524 -77.027921       14.0  2026-03-26T14:17:00   2454020   
4        5506  38.960448 -77.028083        2.0  2026-03-26T14:16:38  17379020   
5        5515  38.944564 -77.026210       11.0  2026-03-26T14:16:55    119020   
6        5501  38.943303 -77.025885       -2.0  2026-03-26T14:17:03  18173020   
7        5532  38.935258 -77.024146        2.0  2026-03-26T14:16:41  11740020   
8        5522  38.901901 -77.021860       10.0  2026-03-26T14:17:03  12357020   
9        5519  38.961555 -77.028027       -4.0  2026-03-26T14:16:50  14335020   
10       5503  38.894574 -77.021919        3.0  2026-03-26T14:16:45  17376020   
11       5536  38.994072 -77

In [17]:
df['timestamp'] = pd.to_datetime(df['timestamp'])
df['trip_start_time'] = pd.to_datetime(df['trip_start_time'])
print(df.dtypes)

vehicle_id                 object
lat                       float64
lon                       float64
deviation                 float64
timestamp          datetime64[ns]
trip_id                    object
route_id                   object
direction                   int64
trip_start_time    datetime64[ns]
agency                     object
dtype: object


In [18]:
routes = ['D40', 'D60', 'D80', 'C53', 'C61', 'D20']
dfs = []

for route in routes:
    wmata_response = requests.get(wmata_url, params={"RouteID": route}, headers=wmata_headers)
    wmata_positions = wmata_response.json()['BusPositions']
    df = pd.DataFrame(wmata_positions)[list(wmata_fields.keys())].rename(columns=wmata_fields)
    df['agency'] = 'WMATA'
    df['timestamp'] = pd.to_datetime(df['timestamp'])
    df['trip_start_time'] = pd.to_datetime(df['trip_start_time'])
    dfs.append(df)

wmata_df = pd.concat(dfs, ignore_index=True)

In [19]:
wmata_df

,vehicle_id,lat,lon,deviation,timestamp,trip_id,route_id,direction,trip_start_time,agency
0,5514,38.993688,-77.030370,5.0,2026-03-26 14:17:01,30634020,D40,0,2026-03-26 13:12:00,WMATA
1,5533,38.895419,-77.021912,2.0,2026-03-26 14:16:42,25675020,D40,1,2026-03-26 13:12:00,WMATA
2,5505,38.915912,-77.021919,6.0,2026-03-26 14:16:39,33133020,D40,1,2026-03-26 13:24:00,WMATA
3,5525,38.961524,-77.027921,14.0,2026-03-26 14:17:00,2454020,D40,0,2026-03-26 13:24:00,WMATA
4,5506,38.960448,-77.028083,2.0,2026-03-26 14:16:38,17379020,D40,0,2026-03-26 13:36:00,WMATA
...,...,...,...,...,...,...,...,...,...,...
61,7204,38.900247,-77.009155,5.0,2026-03-26 14:17:01,10787020,D20,1,2026-03-26 13:50:00,WMATA
62,5484,38.900227,-76.986349,-1.0,2026-03-26 14:16:53,3835020,D20,0,2026-03-26 13:50:00,WMATA
63,5486,38.899818,-77.020211,6.0,2026-03-26 14:16:41,7603020,D20,0,2026-03-26 14:00:00,WMATA
64,5487,38.900222,-76.998013,-1.0,2026-03-26 14:16:53,16076020,D20,1,2026-03-26 14:00:00,WMATA


In [20]:
print(wmata_df['route_id'].unique()) #Checking that all the routes i need are in the df

['D40' 'D60' 'D80' 'C53' 'C61' 'D20']


### 2) MBTA API
#### Bus Routes: Route 1, 23, 28, 39, 57, 66

In [21]:
MBTA_KEY = '4aa18b9edd1341e383dc8fe55a788cca'

In [22]:
#Test
mbta_url = "https://api-v3.mbta.com/vehicles"
mbta_params = {"filter[route]": "1"}
mbta_headers = {"x-api-key": MBTA_KEY}

requests.get(mbta_url, params=mbta_params, headers=mbta_headers)

<Response [200]>

In [41]:
mbta_response = requests.get(mbta_url, params=mbta_params, headers=mbta_headers)
mbta_response.json()

{'data': [{'attributes': {'bearing': 90,
    'carriages': [],
    'current_status': 'STOPPED_AT',
    'current_stop_sequence': 1,
    'direction_id': 0,
    'label': '1922',
    'latitude': 42.329788,
    'longitude': -71.083408,
    'occupancy_status': 'MANY_SEATS_AVAILABLE',
    'revenue': 'REVENUE',
    'speed': None,
    'updated_at': '2026-03-26T14:35:57-04:00'},
   'id': 'y1922',
   'links': {'self': '/vehicles/y1922'},
   'relationships': {'route': {'data': {'id': '1', 'type': 'route'}},
    'stop': {'data': {'id': '64', 'type': 'stop'}},
    'trip': {'data': {'id': '73604109', 'type': 'trip'}}},
   'type': 'vehicle'},
  {'attributes': {'bearing': 340,
    'carriages': [],
    'current_status': 'IN_TRANSIT_TO',
    'current_stop_sequence': 15,
    'direction_id': 0,
    'label': '1902',
    'latitude': 42.353558,
    'longitude': -71.090813,
    'occupancy_status': 'FULL',
    'revenue': 'REVENUE',
    'speed': None,
    'updated_at': '2026-03-26T14:36:01-04:00'},
   'id': 'y190

#### Vars Used
- `id` — vehicle_id
- `attributes.latitude`, `attributes.longitude` — lat, lon
- `attributes.direction_id` — direction (already 0/1, but doesn't tell me which direction that is)
- `attributes.updated_at` — timestamp
- `attributes.current_stop_sequence` — stop_sequence (not available in WMATA)
- `relationships.trip.data.id` — trip_id
- `relationships.route.data.id` — route_id

#### Notes
- No deviation field — must be computed from GTFS static feed in R
- I'll also need GTFS static for the direction north or south bound
- This nested unlike WMATA which was more straight forward

In [35]:
mbta_positions = mbta_response.json()['data']
mbta_positions

mbta_fields = []
for vehicle in mbta_positions:
    row = {
        'vehicle_id': vehicle['id'],
        'lat': vehicle['attributes']['latitude'],
        'lon': vehicle['attributes']['longitude'],
        'direction': vehicle['attributes']['direction_id'],
        'timestamp': vehicle['attributes']['updated_at'],
        'stop_sequence': vehicle['attributes']['current_stop_sequence'],
        'trip_id': vehicle['relationships']['trip']['data']['id'],
        'route_id': vehicle['relationships']['route']['data']['id']
    }
    mbta_fields.append(row)

mbta_df = pd.DataFrame(mbta_fields)
mbta_df['agency'] = 'MBTA'
mbta_df['timestamp'] = pd.to_datetime(mbta_df['timestamp'])

mbta_df

,vehicle_id,lat,lon,direction,timestamp,stop_sequence,trip_id,route_id,agency
0,y1922,42.338384,-71.074515,0,2026-03-26 14:17:19-04:00,1,73604109,1,MBTA
1,y1902,42.336360,-71.076780,0,2026-03-26 14:17:11-04:00,8,73601851,1,MBTA
2,y1888,42.337095,-71.077799,1,2026-03-26 14:17:20-04:00,17,73604085,1,MBTA
3,y1871,42.370020,-71.112870,0,2026-03-26 14:17:11-04:00,22,73604092,1,MBTA
4,y1846,42.371746,-71.117768,1,2026-03-26 14:17:18-04:00,2,73604078,1,MBTA
5,y1828,42.349780,-71.088980,1,2026-03-26 14:17:14-04:00,11,73604095,1,MBTA
6,y1825,42.361379,-71.096935,0,2026-03-26 14:17:19-04:00,17,73604090,1,MBTA
7,y1797,42.332278,-71.081383,0,2026-03-26 14:17:18-04:00,3,73601857,1,MBTA
8,y1753,42.329780,-71.083600,0,2026-03-26 14:17:13-04:00,1,73601853,1,MBTA
9,y1742,42.337651,-71.078332,1,2026-03-26 14:17:19-04:00,17,73604075,1,MBTA


In [40]:
mbta_df.dtypes
mbta_df['timestamp'] = mbta_df['timestamp'].dt.tz_localize(None) # Making time zone consistent
mbta_df.dtypes

vehicle_id               object
lat                     float64
lon                     float64
direction                 int64
timestamp        datetime64[ns]
stop_sequence             int64
trip_id                  object
route_id                 object
agency                   object
dtype: object

In [46]:
mbta_routes = ['1', '23', '28', '39', '57', '66']
mbta_dfs = []

for route in mbta_routes:
    mbta_response = requests.get(mbta_url, params={"filter[route]": route}, headers=mbta_headers)
    mbta_fields = []
    for vehicle in mbta_response.json()['data']:
        row = {
            'vehicle_id': vehicle['id'],
            'lat': vehicle['attributes']['latitude'],
            'lon': vehicle['attributes']['longitude'],
            'direction': vehicle['attributes']['direction_id'],
            'timestamp': vehicle['attributes']['updated_at'],
            'stop_sequence': vehicle['attributes']['current_stop_sequence'],
            'trip_id': vehicle['relationships']['trip']['data']['id'],
            'route_id': vehicle['relationships']['route']['data']['id']
        }
        mbta_fields.append(row)
    mbta_df = pd.DataFrame(mbta_fields)
    mbta_df['agency'] = 'MBTA'
    mbta_df['timestamp'] = pd.to_datetime(mbta_df['timestamp']).dt.tz_localize(None)
    mbta_dfs.append(mbta_df)

mbta_df = pd.concat(mbta_dfs, ignore_index=True)
mbta_df

,vehicle_id,lat,lon,direction,timestamp,stop_sequence,trip_id,route_id,agency
0,y1922,42.333224,-71.073677,0,2026-03-26 14:45:59,6,73604109,1,MBTA
1,y1902,42.365050,-71.103090,0,2026-03-26 14:45:58,18,73601851,1,MBTA
2,y1871,42.341980,-71.083680,1,2026-03-26 14:45:58,15,73604110,1,MBTA
3,y1846,42.340920,-71.082230,1,2026-03-26 14:45:56,16,73604078,1,MBTA
4,y1828,42.332740,-71.080700,1,2026-03-26 14:45:52,22,73604095,1,MBTA
...,...,...,...,...,...,...,...,...,...
72,y3232,42.333160,-71.101040,1,2026-03-26 14:45:38,29,73583989,66,MBTA
73,y3224,42.329851,-71.086364,0,2026-03-26 14:45:56,2,73583873,66,MBTA
74,y3222,42.350587,-71.131303,1,2026-03-26 14:45:32,15,73583995,66,MBTA
75,y3209,42.328899,-71.084717,0,2026-03-26 14:45:53,2,73583871,66,MBTA


### 3) MTA API
#### Bus Routes: B44, B46, B41, Bx12, M14A, M104A

In [48]:
MTA_KEY = '7027c575-3c63-477c-91d8-3d404472e8a0'

In [49]:
mta_url = "http://bustime.mta.info/api/siri/vehicle-monitoring.json"
mta_params = {
    "key": MTA_KEY,
    "LineRef": "MTA NYCT_B44"
}
requests.get(mta_url, params=mta_params)

<Response [200]>

In [50]:
mta_response = requests.get(mta_url, params=mta_params)
mta_response.json()

{'Siri': {'ServiceDelivery': {'ResponseTimestamp': '2026-03-30T18:43:00.161-04:00',
   'VehicleMonitoringDelivery': [{'VehicleActivity': [{'MonitoredVehicleJourney': {'LineRef': 'MTA NYCT_B44',
        'DirectionRef': '0',
        'FramedVehicleJourneyRef': {'DataFrameRef': '2026-03-30',
         'DatedVehicleJourneyRef': 'MTA NYCT_FB_A6-Weekday-SDon-111300_B44_330'},
        'JourneyPatternRef': 'MTA_B44O0113',
        'PublishedLineName': 'B44',
        'OperatorRef': 'MTA NYCT',
        'OriginRef': 'MTA_303440',
        'DestinationName': 'FLUSHING AV via NOSTRAND',
        'SituationRef': [{'SituationSimpleRef': 'MTA NYCT_lmm:planned_work:22527'},
         {'SituationSimpleRef': 'MTA NYCT_lmm:planned_work:15159'}],
        'Monitored': True,
        'VehicleLocation': {'Longitude': -73.944786, 'Latitude': 40.615722},
        'Bearing': 100.95406,
        'ProgressRate': 'normalProgress',
        'BlockRef': 'MTA NYCT_FB_A6-Weekday-SDon_E_FB_43680_B44-331',
        'VehicleRef': 'M

#### Fields Used
- `VehicleRef` →  `"MTA NYCT_"` prefix
- `VehicleLocation.Latitude`, `VehicleLocation.Longitude` 
- `DirectionRef` → direction (comes as string 0/1)
- `RecordedAtTime` → timestamp
- `LineRef` → route_id (strip `"MTA NYCT_"` prefix)
- `MonitoredCall.Extensions.Distances.CallDistanceAlongRoute` → stop_sequence
- `MonitoredCall.ExpectedArrivalTime` - `MonitoredCall.AimedArrivalTime` → used to calc deviation

In [53]:
mta_vehicles = mta_response.json()['Siri']['ServiceDelivery']['VehicleMonitoringDelivery'][0]['VehicleActivity']
mta_vehicles

[{'MonitoredVehicleJourney': {'LineRef': 'MTA NYCT_B44',
   'DirectionRef': '0',
   'FramedVehicleJourneyRef': {'DataFrameRef': '2026-03-30',
    'DatedVehicleJourneyRef': 'MTA NYCT_FB_A6-Weekday-SDon-111300_B44_330'},
   'JourneyPatternRef': 'MTA_B44O0113',
   'PublishedLineName': 'B44',
   'OperatorRef': 'MTA NYCT',
   'OriginRef': 'MTA_303440',
   'DestinationName': 'FLUSHING AV via NOSTRAND',
   'SituationRef': [{'SituationSimpleRef': 'MTA NYCT_lmm:planned_work:22527'},
    {'SituationSimpleRef': 'MTA NYCT_lmm:planned_work:15159'}],
   'Monitored': True,
   'VehicleLocation': {'Longitude': -73.944786, 'Latitude': 40.615722},
   'Bearing': 100.95406,
   'ProgressRate': 'normalProgress',
   'BlockRef': 'MTA NYCT_FB_A6-Weekday-SDon_E_FB_43680_B44-331',
   'VehicleRef': 'MTA NYCT_7936',
   'MonitoredCall': {'AimedArrivalTime': '2026-03-30T18:41:32.000-04:00',
    'ExpectedArrivalTime': '2026-03-30T18:43:01.161-04:00',
    'AimedDepartureTime': '2026-03-30T18:41:32.000-04:00',
    'Expe

In [55]:
print(mta_df.dtypes)

vehicle_id               object
lat                     float64
lon                     float64
direction                 int64
timestamp        datetime64[ns]
route_id                 object
call_distance           float64
deviation_min           float64
agency                   object
dtype: object


In [56]:
mta_rows = []
for vehicle in mta_vehicles:
    mvj = vehicle['MonitoredVehicleJourney']  # the path to data
    mc = mvj['MonitoredCall']  # the path to data
    
    # parse both times so it can be subtracted to get dev
    aimed = pd.to_datetime(mc['AimedArrivalTime'])
    expected = pd.to_datetime(mc['ExpectedArrivalTime'])
    
    row = {
        'vehicle_id': mvj['VehicleRef'].replace('MTA NYCT_', ''),  # removing prefix
        'lat': mvj['VehicleLocation']['Latitude'],
        'lon': mvj['VehicleLocation']['Longitude'],
        'direction': int(mvj['DirectionRef']),  # making the string "0"/"1" a int
        'timestamp': pd.to_datetime(vehicle['RecordedAtTime']),
        'route_id': mvj['LineRef'].replace('MTA NYCT_', ''),  # removing prefix
        'call_distance': mc['Extensions']['Distances']['CallDistanceAlongRoute'],  # stop_sequence
        'deviation_min': (expected - aimed).total_seconds() / 60  # positive = late, negative = early
    }
    mta_rows.append(row)

mta_df = pd.DataFrame(mta_rows)
mta_df['agency'] = 'MTA'
mta_df['timestamp'] = mta_df['timestamp'].dt.tz_localize(None)  # making timezone to match WMATA/MBTA
mta_df

,vehicle_id,lat,lon,direction,timestamp,route_id,call_distance,deviation_min,agency
0,7936,40.615722,-73.944786,0,2026-03-30 18:42:36.129,B44,1670.59,1.486017,MTA
1,7340,40.596122,-73.941050,1,2026-03-30 18:42:56.000,B44,11633.46,11.336017,MTA
2,7947,40.636698,-73.945155,0,2026-03-30 18:42:53.000,B44,4248.88,5.519883,MTA
3,7701,40.621674,-73.945902,0,2026-03-30 18:42:49.000,B44,4983.21,7.352683,MTA
4,7942,40.694243,-73.956062,0,2026-03-30 18:42:41.000,B44,13840.12,5.352683,MTA
5,7618,40.691024,-73.955424,0,2026-03-30 18:42:40.000,B44,10849.80,0.036017,MTA
6,7705,40.645373,-73.948985,1,2026-03-30 18:42:43.000,B44,6166.82,-1.338667,MTA
7,7945,40.667473,-73.950708,1,2026-03-30 18:42:37.835,B44,3705.61,-0.727150,MTA
8,7641,40.686259,-73.950754,1,2026-03-30 18:42:52.000,B44,1615.56,-1.179933,MTA
9,7656,40.625080,-73.946537,1,2026-03-30 18:42:41.000,B44,8462.38,-8.817933,MTA


In [68]:
mta_routes = ['B44', 'B46', 'B41', 'Bx12', 'M14A', 'M104']
mta_dfs = []

for route in mta_routes:
    mta_params = {"key": MTA_KEY, "LineRef": f"MTA NYCT_{route}"} #This needs to be in the loop not outside
    response = requests.get(mta_url, params=mta_params)
    delivery = response.json()['Siri']['ServiceDelivery']['VehicleMonitoringDelivery'][0]
    mta_vehicles = delivery.get('VehicleActivity', [])  # creating an empty list if no active vehicles (this caused loop to not work)
    
    mta_rows = []
    for vehicle in mta_vehicles:
        mvj = vehicle['MonitoredVehicleJourney']  # the path to data
        if 'MonitoredCall' not in mvj:  # skip buses in layover (this also caused loop to not work)
            continue
        mc = mvj['MonitoredCall']  # the path to data

        if 'ExpectedArrivalTime' not in mc:  # skip vehicles with no prediction (Dropping, won't be able to calc dev)
            continue
        
         # parse both times so it can be subtracted to get dev
        aimed = pd.to_datetime(mc['AimedArrivalTime'])
        expected = pd.to_datetime(mc['ExpectedArrivalTime'])
        
        row = {
            'vehicle_id': mvj['VehicleRef'].replace('MTA NYCT_', ''),  # removing prefix
            'lat': mvj['VehicleLocation']['Latitude'],
            'lon': mvj['VehicleLocation']['Longitude'],
            'direction': int(mvj['DirectionRef']),  # making the string "0"/"1" a int
            'timestamp': pd.to_datetime(vehicle['RecordedAtTime']),
            'route_id': mvj['LineRef'].replace('MTA NYCT_', ''),  # removing prefix
            'call_distance': mc['Extensions']['Distances']['CallDistanceAlongRoute'],  # stop_sequence equivalent
            'deviation_min': (expected - aimed).total_seconds() / 60  # positive = late, negative = early
        }
        mta_rows.append(row)
    
    df = pd.DataFrame(mta_rows)
    if df.empty:  # skip if no valid vehicles for this route
        continue
    df['agency'] = 'MTA'
    df['timestamp'] = df['timestamp'].dt.tz_localize(None)  # making timezone to match WMATA/MBTA
    mta_dfs.append(df)

mta_df = pd.concat(mta_dfs, ignore_index=True)
mta_df

,vehicle_id,lat,lon,direction,timestamp,route_id,call_distance,deviation_min,agency
0,7942,40.640676,-73.948471,1,2026-03-30 19:34:54.000,B44,6630.35,1.016233,MTA
1,7654,40.621837,-73.945933,0,2026-03-30 19:34:58.000,B44,4983.21,0.682900,MTA
2,7641,40.600601,-73.941913,1,2026-03-30 19:34:43.000,B44,11123.96,-4.983767,MTA
3,7941,40.668573,-73.947844,0,2026-03-30 19:34:57.017,B44,7845.65,-1.856300,MTA
4,7936,40.692795,-73.955765,0,2026-03-30 19:34:36.129,B44,11084.16,-3.067100,MTA
...,...,...,...,...,...,...,...,...,...
65,9854,40.794087,-73.972324,1,2026-03-30 19:34:44.537,M104,3778.31,-0.149600,MTA
66,9855,40.815415,-73.958061,1,2026-03-30 19:34:47.943,M104,1053.73,5.371450,MTA
67,4690,40.784941,-73.979261,1,2026-03-30 19:34:38.000,M104,5007.25,6.305550,MTA
68,9898,40.760183,-73.987603,0,2026-03-30 19:34:32.644,M104,694.92,7.179483,MTA


In [75]:
print(mta_df.groupby('route_id').size())

route_id
B41     31
B44     14
B46     16
M104     9
dtype: int64


In [72]:
# Bx12 and M14 missing

requests.get(mta_url, params={"key": MTA_KEY, "LineRef": "MTA NYCT_Bx12"})
requests.get(mta_url, params={"key": MTA_KEY, "LineRef": "MTA NYCT_M14A"})


<Response [200]>

In [73]:
bx12_response = requests.get(mta_url, params={"key": MTA_KEY, "LineRef": "MTA NYCT_Bx12"})
print(bx12_response.json()['Siri']['ServiceDelivery']['VehicleMonitoringDelivery'][0])

{'ResponseTimestamp': '2026-03-30T19:38:00.628-04:00', 'ErrorCondition': {'OtherError': {'ErrorText': 'No such route: MTA NYCT_Bx12.'}, 'Description': 'No such route: MTA NYCT_Bx12.'}}


In [74]:
m14a_response = requests.get(mta_url, params={"key": MTA_KEY, "LineRef": "MTA NYCT_M14A"})
print(m14a_response.json()['Siri']['ServiceDelivery']['VehicleMonitoringDelivery'][0])

{'ResponseTimestamp': '2026-03-30T19:39:04.108-04:00', 'ErrorCondition': {'OtherError': {'ErrorText': 'No such route: MTA NYCT_M14A.'}, 'Description': 'No such route: MTA NYCT_M14A.'}}


In [77]:
test_response = requests.get(
    "http://bustime.mta.info/api/where/routes-for-agency/MTA%20NYCT.json",
    params={"key": MTA_KEY}
)
routes = test_response.json()['data']['list']
for r in routes:
    if 'x12' in r['id'].lower() or 'm14' in r['id'].lower():
        print(r['id'], r['shortName'])

MTA NYCT_BX12+ Bx12-SBS
MTA NYCT_BX12 Bx12
MTA NYCT_M14A+ M14A-SBS
MTA NYCT_M14D+ M14D-SBS


In [83]:
mta_routes = ['B44', 'B46', 'B41', 'BX12', 'M14A+', 'M104']
mta_dfs = []

for route in mta_routes:
    mta_params = {"key": MTA_KEY, "LineRef": f"MTA NYCT_{route}"} #This needs to be in the loop not outside
    response = requests.get(mta_url, params=mta_params)
    delivery = response.json()['Siri']['ServiceDelivery']['VehicleMonitoringDelivery'][0]
    mta_vehicles = delivery.get('VehicleActivity', [])  # creating an empty list if no active vehicles (this caused loop to not work)
    
    mta_rows = []
    for vehicle in mta_vehicles:
        mvj = vehicle['MonitoredVehicleJourney']  # the path to data
        if 'MonitoredCall' not in mvj:  # skip buses in layover (this also caused loop to not work)
            continue
        mc = mvj['MonitoredCall']  # the path to data

        if 'ExpectedArrivalTime' not in mc:  # skip vehicles with no prediction (Dropping, won't be able to calc dev)
            continue
        
         # parse both times so it can be subtracted to get dev
        aimed = pd.to_datetime(mc['AimedArrivalTime'])
        expected = pd.to_datetime(mc['ExpectedArrivalTime'])
        
        row = {
            'vehicle_id': mvj['VehicleRef'].replace('MTA NYCT_', ''),  # removing prefix
            'lat': mvj['VehicleLocation']['Latitude'],
            'lon': mvj['VehicleLocation']['Longitude'],
            'direction': int(mvj['DirectionRef']),  # making the string "0"/"1" a int
            'timestamp': pd.to_datetime(vehicle['RecordedAtTime']),
            'route_id': mvj['LineRef'].replace('MTA NYCT_', ''),  # removing prefix
            'call_distance': mc['Extensions']['Distances']['CallDistanceAlongRoute'],  # stop_sequence equivalent
            'deviation': (expected - aimed).total_seconds() / 60  # positive = late, negative = early
        }
        mta_rows.append(row)
    
    df = pd.DataFrame(mta_rows)
    if df.empty:  # skip if no valid vehicles for this route
        continue
    df['agency'] = 'MTA'
    df['timestamp'] = df['timestamp'].dt.tz_localize(None)  # making timezone to match WMATA/MBTA
    mta_dfs.append(df)

mta_df = pd.concat(mta_dfs, ignore_index=True)
mta_df

,vehicle_id,lat,lon,direction,timestamp,route_id,call_distance,deviation,agency
0,7342,40.641016,-73.945615,0,2026-03-30 20:25:15.000,B44,7347.40,0.338967,MTA
1,7641,40.653017,-73.946900,0,2026-03-30 20:25:07.000,B44,6065.93,1.155633,MTA
2,7942,40.615547,-73.944752,0,2026-03-30 20:25:08.000,B44,1670.59,5.705633,MTA
3,7936,40.666037,-73.950833,1,2026-03-30 20:25:00.319,B44,3862.50,-7.230783,MTA
4,7945,40.680357,-73.949083,0,2026-03-30 20:25:07.835,B44,9365.33,-0.305683,MTA
...,...,...,...,...,...,...,...,...,...
66,9792,40.788204,-73.976696,1,2026-03-30 20:25:01.000,M104,4432.17,2.707183,MTA
67,9640,40.769748,-73.982219,1,2026-03-30 20:25:09.803,M104,6596.91,-1.076150,MTA
68,9855,40.761027,-73.983249,1,2026-03-30 20:25:17.944,M104,7913.95,3.274300,MTA
69,9808,40.805073,-73.966147,1,2026-03-30 20:25:10.717,M104,2443.43,0.330817,MTA


In [79]:
print(mta_df.groupby('route_id').size())

route_id
B41      31
B44      14
B46      14
BX12      6
M104      8
M14A+     7
dtype: int64


In [84]:
# Checking that all dfs have the came columns

print("WMATA columns:", wmata_df.columns.tolist())
print("MBTA columns:", mbta_df.columns.tolist())
print("MTA columns:", mta_df.columns.tolist())

WMATA columns: ['vehicle_id', 'lat', 'lon', 'deviation', 'timestamp', 'trip_id', 'route_id', 'direction', 'trip_start_time', 'agency']
MBTA columns: ['vehicle_id', 'lat', 'lon', 'direction', 'timestamp', 'stop_sequence', 'trip_id', 'route_id', 'agency']
MTA columns: ['vehicle_id', 'lat', 'lon', 'direction', 'timestamp', 'route_id', 'call_distance', 'deviation', 'agency']


In [88]:
#Updated Routes
wmata_routes = ['D40', 'D60', 'D80', 'C53', 'C61', 'D20', 'P40', 'C81', 'P16', 'D24', 'P14', 'C23', 'P20', 'C35']
mbta_routes = ['1', '23', '28', '39', '57', '66', '111', '64', '201', '50', '42', '87', '85', '70']
mta_routes = ['B44', 'B46', 'B41', 'BX12', 'M14A+', 'M23+', 'Q98', 'B32', 'M21', 'BX42', 'M50', 'Q67', 'B84', 'Q42']

In [89]:
wmata_fields = {
    'VehicleID': 'vehicle_id',
    'Lat': 'lat',
    'Lon': 'lon',
    'Deviation': 'deviation',
    'DateTime': 'timestamp',
    'TripID': 'trip_id',
    'RouteID': 'route_id',
    'DirectionNum': 'direction',
    'TripStartTime': 'trip_start_time'
}

In [103]:
import time
import os

output_dir = "data"
os.makedirs(output_dir, exist_ok=True)  # creating data folder to store csv

wmata_file = os.path.join(output_dir, "wmata.csv")
mbta_file = os.path.join(output_dir, "mbta.csv")
mta_file = os.path.join(output_dir, "mta.csv")

while True:
    try:
# WMATA
        try: #this allows for the loop to contuine if the pull fails for one agency
            wmata_dfs = []
            for route in wmata_routes:
                wmata_response = requests.get(wmata_url, params={"RouteID": route}, headers=wmata_headers)
                wmata_positions = wmata_response.json()['BusPositions']
                if not wmata_positions:  # skip if no active vehicles
                    continue
                df = pd.DataFrame(wmata_positions)[list(wmata_fields.keys())].rename(columns=wmata_fields)
                df['agency'] = 'WMATA'
                df['timestamp'] = pd.to_datetime(df['timestamp'])
                df['trip_start_time'] = pd.to_datetime(df['trip_start_time'])
                wmata_dfs.append(df)
            if wmata_dfs:
                wmata_df = pd.concat(wmata_dfs, ignore_index=True)
                wmata_df.to_csv(wmata_file, mode='a', header=not os.path.exists(wmata_file), index=False) # appending pulls to exsiting file
        except Exception as e:
            print(f"WMATA error: {e}") #if an error this will print it and move on

# MBTA
        try:
            mbta_dfs = []
            for route in mbta_routes:
                mbta_response = requests.get(mbta_url, params={"filter[route]": route}, headers=mbta_headers)
                mbta_vehicles = mbta_response.json().get('data', [])  # empty list if no active vehicles
                if not mbta_vehicles:  # skip if no active vehicles
                    continue
                mbta_fields = []
                for vehicle in mbta_vehicles:
                    row = {
                        'vehicle_id': vehicle['id'],
                        'lat': vehicle['attributes']['latitude'],
                        'lon': vehicle['attributes']['longitude'],
                        'direction': vehicle['attributes']['direction_id'],
                        'timestamp': vehicle['attributes']['updated_at'],
                        'stop_sequence': vehicle['attributes']['current_stop_sequence'],
                        'trip_id': vehicle['relationships']['trip']['data']['id'],
                        'route_id': vehicle['relationships']['route']['data']['id']
                    }
                    mbta_fields.append(row)
                df = pd.DataFrame(mbta_fields)
                if df.empty:
                    continue
                df['agency'] = 'MBTA'
                df['timestamp'] = pd.to_datetime(df['timestamp']).dt.tz_localize(None)
                mbta_dfs.append(df)
            if mbta_dfs:
                mbta_df = pd.concat(mbta_dfs, ignore_index=True)
                mbta_df.to_csv(mbta_file, mode='a', header=not os.path.exists(mbta_file), index=False)
        except Exception as e:
            print(f"MBTA error: {e}")

# MTA
        try:
            mta_dfs = []
            for route in mta_routes:
                mta_params = {"key": MTA_KEY, "LineRef": f"MTA NYCT_{route}"}  # needs to be in the loop
                response = requests.get(mta_url, params=mta_params)
                delivery = response.json()['Siri']['ServiceDelivery']['VehicleMonitoringDelivery'][0]
                mta_vehicles = delivery.get('VehicleActivity', [])  # empty list if no active vehicles
                mta_rows = []
                for vehicle in mta_vehicles:
                    mvj = vehicle['MonitoredVehicleJourney']  # the path to data
                    if 'MonitoredCall' not in mvj:  # skip buses in layover
                        continue
                    mc = mvj['MonitoredCall']  # the path to data
                    if 'ExpectedArrivalTime' not in mc:  # skip vehicles with no prediction
                        continue
                    # parse both times so it can be subtracted to get dev
                    aimed = pd.to_datetime(mc['AimedArrivalTime'])
                    expected = pd.to_datetime(mc['ExpectedArrivalTime'])
                    row = {
                        'vehicle_id': mvj['VehicleRef'].replace('MTA NYCT_', ''),  # removing prefix
                        'lat': mvj['VehicleLocation']['Latitude'],
                        'lon': mvj['VehicleLocation']['Longitude'],
                        'direction': int(mvj['DirectionRef']),  # making the string "0"/"1" to int
                        'timestamp': pd.to_datetime(vehicle['RecordedAtTime']),
                        'route_id': mvj['LineRef'].replace('MTA NYCT_', ''),  # removing prefix
                        'call_distance': mc['Extensions']['Distances']['CallDistanceAlongRoute'],  # stop_sequence equivalent
                        'deviation': (expected - aimed).total_seconds() / 60  # positive = late, negative = early
                    }
                    mta_rows.append(row)
                df = pd.DataFrame(mta_rows)
                if df.empty:  # skip if no valid vehicles for this route
                    continue
                df['agency'] = 'MTA'
                df['timestamp'] = df['timestamp'].dt.tz_localize(None)  # making timezone to match WMATA/MBTA
                mta_dfs.append(df)
            if mta_dfs:
                mta_df = pd.concat(mta_dfs, ignore_index=True)
                mta_df.to_csv(mta_file, mode='a', header=not os.path.exists(mta_file), index=False)
        except Exception as e:
            print(f"MTA error: {e}")

        poll_count += 1
        print(f"Poll complete: {pd.Timestamp.now().strftime('%H:%M:%S')} (poll {poll_count})")
        
        if poll_count % 5 == 0:  # every 5 polls save notebook and push to github
            os.system('jupyter nbconvert --to notebook --inplace "API Project.ipynb"')
            os.system('git add data/ "API Project.ipynb" && git commit -m "auto: update data poll {poll_count}" && git push origin main')
            print(f"Pushed to GitHub at poll {poll_count}")
            
        time.sleep(60)  # wait 1 minute before next poll

    except KeyboardInterrupt:
        print("Polling stopped.")
        break

Poll complete: 21:05:03
Polling stopped.


In [104]:
# Checking how many rows and columns in each df
pd.read_csv("data/wmata.csv").shape, pd.read_csv("data/mbta.csv").shape, pd.read_csv("data/mta.csv").shape

((43719, 10), (39423, 9), (49794, 9))

In [97]:
response = requests.get(wmata_url, params={"RouteID": "P20"}, headers=wmata_headers)
print(response.json())

{'BusPositions': []}


In [98]:
response = requests.get(
    "https://api.wmata.com/Bus.svc/json/jRoutes",
    headers=wmata_headers
)
routes = response.json()['Routes']
for r in routes:
    if 'P20' in r['RouteID']:
        print(r)

{'RouteID': 'P20', 'Name': 'P20 - NEW CRLTN - GREENBELT', 'LineDescription': 'Greenbelt Rd-New Carrollton'}
